In [ ]:

import torch
import torch.nn as nn
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Re = 100.0
n_collocation = 500
epochs = 500
lr = 1e-3

In [ ]:
class QLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.randn(6))
        self.params = ParameterVector("θ", 6)

        qc = QuantumCircuit(2)
        qc.ry(self.params[0], 0)
        qc.ry(self.params[1], 1)
        qc.cz(0,1)
        qc.ry(self.params[2], 0)
        qc.ry(self.params[3], 1)
        qc.cz(0,1)
        qc.ry(self.params[4], 0)
        qc.ry(self.params[5], 1)

        self.circuit = qc

    def forward(self, x):
        outputs = []
        for xi in x:
            theta_vals = self.theta.detach().cpu().numpy()
            job = self.estimator.run(self.circuit, [self.circuit], [theta_vals]) # type: ignore
            result = job.result().values[0]
            outputs.append([result])
        return torch.tensor(outputs, dtype=torch.float32).to(next(self.parameters()).device)

In [ ]:
class QPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = nn.Linear(2, 2)
        self.quantum = QLayer()
        self.post = nn.Linear(1, 3)

    def forward(self, x):
        x = torch.tanh(self.pre(x))
        q_out = self.quantum(x)
        return self.post(q_out)